In [9]:
#!/usr/bin/env python3
"""
=============================================================================
재실험코드 6 (Notebook 06) — Supplementary Analyses
=============================================================================

이 스크립트는 재실험코드 3/4의 기존 rank cache와 결과를 활용하여
추가 분석을 수행합니다. 임베딩/검색을 재실행하지 않습니다.

이슈 1: Query-source term overlap 측정 + Lexical shortcut 서브셋 분석
이슈 2: Paired bootstrap 통계 검정 (Hit@10 차이의 95% CI, p-value)
이슈 3: PatentSBERTa 2-way grid search (equal-weight bias 검증)
이슈 4: WIPO 5대 섹터별 difficulty decomposition

Hybrid scoring: 재실험코드 3과 동일한 Weighted RRF (K_RRF=60) 사용.
예상 실행 시간: 전체 약 5~10분 (GPU 불필요)
=============================================================================
"""

import os
import json
import hashlib
import re
from collections import Counter

import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# 사용자 환경에 맞게 아래 경로를 수정하세요
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR        = "2nd_exp"
CACHE_ROOT      = os.path.join(BASE_DIR, "cache", "notebook03_rankcache_fullcorpus")
DATA_DIR        = os.path.join(BASE_DIR, "data")
RESULTS_DIR_NB3 = os.path.join(BASE_DIR, "results", "notebook03_eval_v2")
OUT_DIR         = os.path.join(BASE_DIR, "results", "notebook06_supplementary")
os.makedirs(OUT_DIR, exist_ok=True)

# 코퍼스 CSV 파일명 패턴 (연도별)
CORPUS_PATTERN = "independent_claims_y{year}_with_abstract_wipo_cpc.csv"

YEARS       = [2005, 2015, 2025]
QUERY_TYPES = ["summary", "keyword", "function"]
TOP_L       = 100   # rank cache의 top-L
K_RRF       = 60    # RRF constant (재실험코드 3과 동일)

# ─────────────────────────────────────────────────────────────────────────────
# 10개 모델 정의 (재실험코드 3과 동일)
# ─────────────────────────────────────────────────────────────────────────────
MODELS = [
    # (model_id, display_name, kind, field, encoder, max_length)
    ("BM25_CLAIMS",                   "BM25_Claims",                   "bm25",  "claims",   None,                        None),
    ("BM25_ABSTRACT",                 "BM25_Abstract",                 "bm25",  "abstract", None,                        None),
    ("PATENTSBERTA_CLAIMS_DENSE",     "PatentSBERTa_Claims_Dense",     "dense", "claims",   "AI-Growth-Lab/PatentSBERTa", 512),
    ("PATENTSBERTA_ABSTRACT_DENSE",   "PatentSBERTa_Abstract_Dense",   "dense", "abstract", "AI-Growth-Lab/PatentSBERTa", 512),
    ("PATENTSBERTA_CLAIMS_HYBRID",    "PatentSBERTa_Claims_Hybrid",    "hybrid","claims",   "AI-Growth-Lab/PatentSBERTa", 512),
    ("PATENTSBERTA_ABSTRACT_HYBRID",  "PatentSBERTa_Abstract_Hybrid",  "hybrid","abstract", "AI-Growth-Lab/PatentSBERTa", 512),
    ("BGE_M3_512_CLAIMS_DENSE",       "BGE-M3_512_Claims_Dense",       "dense", "claims",   "BAAI/bge-m3",                512),
    ("BGE_M3_512_ABSTRACT_DENSE",     "BGE-M3_512_Abstract_Dense",     "dense", "abstract", "BAAI/bge-m3",                512),
    ("BGE_M3_512_CLAIMS_HYBRID",      "BGE-M3_512_Claims_Hybrid",      "hybrid","claims",   "BAAI/bge-m3",                512),
    ("BGE_M3_512_ABSTRACT_HYBRID",    "BGE-M3_512_Abstract_Hybrid",    "hybrid","abstract", "BAAI/bge-m3",                512),
]

# hybrid 모델의 구성요소 매핑 (dense rank cache + bm25 rank cache)
# BM25-only와 Dense-only는 rank cache 하나만 사용
def safe_name(x):
    """encoder 이름을 파일명에 안전한 형태로 변환 (재실험코드 3의 safe_name과 동일)"""
    if x is None:
        return None
    return re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(x))

def _rank_cache_path(year, kind, field, encoder=None, max_length=None):
    """rank cache .ranks.npy 파일 경로 생성 (재실험코드 3의 rankcache_paths와 동일 로직)"""
    ydir = os.path.join(CACHE_ROOT, str(year), "rankcache")
    if kind == "bm25":
        fname = f"bm25__{field}__L{TOP_L}.ranks.npy"
    else:
        enc = safe_name(encoder)
        fname = f"dense__{field}__{enc}__ML{int(max_length)}__L{TOP_L}.ranks.npy"
    return os.path.join(ydir, fname)


# =============================================================================
# 공통: 데이터 로딩
# =============================================================================
print("=" * 70)
print("재실험코드 6 (NOTEBOOK 06) — Supplementary Analyses")
print("=" * 70)

def load_queries(year):
    """queries_final.csv 로드 (재실험코드 3과 동일한 정렬 적용)"""
    path = os.path.join(DATA_DIR, str(year), "queries_final.csv")
    if not os.path.exists(path):
        # fallback
        path2 = os.path.join(DATA_DIR, str(year), "queries.csv")
        if os.path.exists(path2):
            path = path2
        else:
            raise FileNotFoundError(f"[{year}] queries file not found")
    
    dfq = pd.read_csv(path, dtype={"patent_id": str})
    dfq["patent_id"] = dfq["patent_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    dfq["query_type"] = dfq["query_type"].astype(str).str.lower().str.strip()
    dfq["query_text"] = dfq["query_text"].fillna("").astype(str)
    
    dfq = dfq[dfq["query_type"].isin(QUERY_TYPES)].reset_index(drop=True)
    
    # 재실험코드 3과 동일: query_key 생성 후 정렬
    if "query_key" not in dfq.columns:
        dfq["query_key"] = [f"{year}::{i}" for i in range(len(dfq))]
    
    # *** 핵심: 재실험코드 3의 L242와 동일한 정렬 ***
    dfq = dfq.sort_values(["query_type", "patent_id", "query_key"]).reset_index(drop=True)
    
    return dfq

def load_corpus(year):
    """전체 코퍼스 CSV 로드 (patent_id, claims, abstract, wipo_field)"""
    fname = CORPUS_PATTERN.format(year=year)
    # 코퍼스 파일 위치를 여러 후보에서 탐색
    candidates = [
        os.path.join(DATA_DIR, str(year), fname),
        os.path.join(BASE_DIR, fname),
        fname,  # 현재 디렉토리
    ]
    for path in candidates:
        if os.path.exists(path):
            df = pd.read_csv(path, dtype={"patent_id": str})
            df["patent_id"] = df["patent_id"].str.strip().str.replace(r"\.0$", "", regex=True)
            return df
    raise FileNotFoundError(f"Corpus file not found for year={year}. Tried: {candidates}")

def build_gt_index(dfq, corpus_pids):
    """각 쿼리의 ground truth patent_id → 코퍼스 내 doc index 매핑"""
    pid_to_idx = {pid: i for i, pid in enumerate(corpus_pids)}
    gt_indices = []
    valid_mask = []
    for pid in dfq["patent_id"]:
        if pid in pid_to_idx:
            gt_indices.append(pid_to_idx[pid])
            valid_mask.append(True)
        else:
            gt_indices.append(-1)
            valid_mask.append(False)
    return np.array(gt_indices, dtype=np.int64), np.array(valid_mask, dtype=bool)


# =============================================================================
# 공통: query-level hit 계산 (재실험코드 3과 동일한 patent_id 문자열 비교 방식)
# =============================================================================
def hit_at_k(ranked_pids, gold_pid, k):
    """재실험코드 3의 hit_at_k와 동일"""
    return 1.0 if gold_pid in ranked_pids[:k] else 0.0

def mrr_at_k(ranked_pids, gold_pid, k):
    """재실험코드 3의 mrr_at_k와 동일"""
    for i, pid in enumerate(ranked_pids[:k], start=1):
        if pid == gold_pid:
            return 1.0 / i
    return 0.0

def compute_metrics_for_query(ranked_doc_indices, gold_pid, corpus_pids):
    """
    rank cache의 정수 doc index를 patent_id 문자열로 변환한 뒤
    gold_pid와 문자열 비교. 재실험코드 3의 compute_metrics 로직과 동일.
    """
    ranked_pids = [corpus_pids[int(d)] for d in ranked_doc_indices]
    return {
        "Hit@10": hit_at_k(ranked_pids, gold_pid, 10),
        "MRR@10": mrr_at_k(ranked_pids, gold_pid, 10),
        "Hit@50": hit_at_k(ranked_pids, gold_pid, 50),
    }

def weighted_rrf_fuse(dense_rank, bm25_rank, w_dense, w_sparse, k_rrf, out_topk):
    """
    재실험코드 3의 weighted_rrf_fuse와 동일한 구현.
    Weighted Reciprocal Rank Fusion으로 dense + BM25 rank를 결합.
    score(doc) = w_dense / (k_rrf + rank_in_dense) + w_sparse / (k_rrf + rank_in_bm25)
    """
    scores = {}
    for r, doc in enumerate(dense_rank, start=1):
        scores[int(doc)] = scores.get(int(doc), 0.0) + (w_dense / (k_rrf + r))
    for r, doc in enumerate(bm25_rank, start=1):
        scores[int(doc)] = scores.get(int(doc), 0.0) + (w_sparse / (k_rrf + r))
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return np.array([doc for doc, _ in ranked[:out_topk]], dtype=np.int32)

def get_hybrid_top(dense_ranks, bm25_ranks, w_dense, w_bm25, out_topk=50):
    """
    전체 쿼리에 대해 weighted_rrf_fuse를 적용하여 hybrid top-K doc indices 반환.
    dense_ranks, bm25_ranks: (n_queries, TOP_L) shape의 rank cache
    Returns: (n_queries, out_topk) shape의 fused rank array
    """
    n_q = dense_ranks.shape[0]
    hybrid_top = np.zeros((n_q, out_topk), dtype=np.int32)
    for i in range(n_q):
        hybrid_top[i] = weighted_rrf_fuse(
            dense_ranks[i], bm25_ranks[i],
            w_dense, w_bm25, K_RRF, out_topk
        )
    return hybrid_top


# =============================================================================
# Phase 0: 모든 연도의 query-level hit 매트릭스 구축
# =============================================================================
print("\n" + "=" * 70)
print("Phase 0: Building query-level hit matrix from rank caches")
print("=" * 70)

# 결과 저장 구조: {year: {model_id: {"hit10": array, "hit50": array, "mrr10": array}}}
all_query_hits = {}
all_query_meta = {}  # {year: DataFrame with patent_id, query_type, wipo, etc.}

for year in YEARS:
    print(f"\n--- Year {year} ---")
    
    # 쿼리 로드
    dfq = load_queries(year)
    print(f"  Queries loaded: {len(dfq)}")
    
    # 코퍼스 로드 — patent_id 리스트가 rank cache 생성 시와 동일해야 함
    corpus = load_corpus(year)
    # 재실험코드 3과 동일하게 duplicate 제거 (keep first) + reset_index
    corpus = corpus.drop_duplicates(subset="patent_id", keep="first").reset_index(drop=True)
    corpus_pids = corpus["patent_id"].tolist()
    print(f"  Corpus loaded: {len(corpus_pids):,} docs (after dedup)")
    
    # gold patent_id 목록 (쿼리별)
    gold_pids = dfq["patent_id"].astype(str).tolist()
    
    # valid mask: gold_pid가 코퍼스에 존재하는지
    pid_set = set(corpus_pids)
    valid_mask = np.array([pid in pid_set for pid in gold_pids], dtype=bool)
    n_valid = valid_mask.sum()
    print(f"  Valid queries: {n_valid} / {len(dfq)}")
    
    # 쿼리 메타데이터 (query_type, patent_id, wipo 등)
    dfq_meta = dfq[["patent_id", "query_type"]].copy()
    dfq_meta["query_text"] = dfq["query_text"] if "query_text" in dfq.columns else ""
    dfq_meta["year"] = year
    dfq_meta["valid"] = valid_mask
    
    # wipo 컬럼: queries_final.csv에 "wipo" 컬럼이 있으면 직접 사용
    if "wipo" in dfq.columns:
        dfq_meta["wipo_field"] = dfq["wipo"].values
    else:
        pid_to_wipo = {}
        wipo_col = None
        for col in ["wipo_field", "wipo", "WIPO_field", "wipo_code"]:
            if col in corpus.columns:
                wipo_col = col
                break
        if wipo_col:
            pid_to_wipo = dict(zip(corpus["patent_id"], corpus[wipo_col]))
            dfq_meta["wipo_field"] = dfq_meta["patent_id"].map(pid_to_wipo)
    
    # source patent의 claims/abstract 텍스트 (이슈 1: overlap 계산용)
    claims_col = None
    abstract_col = None
    for col in ["claims", "independent_claims", "Claims"]:
        if col in corpus.columns:
            claims_col = col
            break
    for col in ["abstract", "Abstract"]:
        if col in corpus.columns:
            abstract_col = col
            break
    
    if claims_col:
        pid_to_claims = dict(zip(corpus["patent_id"],
                                  corpus[claims_col].fillna("").astype(str)))
    else:
        pid_to_claims = {}
    if abstract_col:
        pid_to_abstract = dict(zip(corpus["patent_id"],
                                    corpus[abstract_col].fillna("").astype(str)))
    else:
        pid_to_abstract = {}
    
    dfq_meta["source_claims"] = dfq_meta["patent_id"].map(pid_to_claims).fillna("")
    dfq_meta["source_abstract"] = dfq_meta["patent_id"].map(pid_to_abstract).fillna("")
    
    all_query_meta[year] = dfq_meta
    
    # ─── 각 모델의 query-level hit 계산 (patent_id 문자열 비교) ───
    year_hits = {}
    n_queries = len(dfq)
    
    for model_id, display_name, kind, field, encoder, max_length in MODELS:
        
        per_hit10 = np.full(n_queries, -1.0)
        per_mrr10 = np.full(n_queries, -1.0)
        per_hit50 = np.full(n_queries, -1.0)
        
        if kind == "bm25":
            rc_path = _rank_cache_path(year, "bm25", field)
            if not os.path.exists(rc_path):
                print(f"  [WARN] rank cache not found: {rc_path}")
                continue
            ranks = np.load(rc_path)  # (3150, 100)
            
            for i in range(n_queries):
                if not valid_mask[i]:
                    continue
                # ranks[i]의 정수 doc index → patent_id 문자열로 변환 후 비교
                ranked_pids = [corpus_pids[int(d)] for d in ranks[i, :50]]
                m = {"Hit@10": hit_at_k(ranked_pids, gold_pids[i], 10),
                     "MRR@10": mrr_at_k(ranked_pids, gold_pids[i], 10),
                     "Hit@50": hit_at_k(ranked_pids, gold_pids[i], 50)}
                per_hit10[i] = m["Hit@10"]
                per_mrr10[i] = m["MRR@10"]
                per_hit50[i] = m["Hit@50"]
            
        elif kind == "dense":
            rc_path = _rank_cache_path(year, "dense", field, encoder, max_length)
            if not os.path.exists(rc_path):
                print(f"  [WARN] rank cache not found: {rc_path}")
                continue
            ranks = np.load(rc_path)
            
            for i in range(n_queries):
                if not valid_mask[i]:
                    continue
                ranked_pids = [corpus_pids[int(d)] for d in ranks[i, :50]]
                m = {"Hit@10": hit_at_k(ranked_pids, gold_pids[i], 10),
                     "MRR@10": mrr_at_k(ranked_pids, gold_pids[i], 10),
                     "Hit@50": hit_at_k(ranked_pids, gold_pids[i], 50)}
                per_hit10[i] = m["Hit@10"]
                per_mrr10[i] = m["MRR@10"]
                per_hit50[i] = m["Hit@50"]
            
        elif kind == "hybrid":
            dense_rc = _rank_cache_path(year, "dense", field, encoder, max_length)
            bm25_rc  = _rank_cache_path(year, "bm25", field)
            if not os.path.exists(dense_rc) or not os.path.exists(bm25_rc):
                print(f"  [WARN] rank cache not found for hybrid: {display_name}")
                continue
            dense_ranks_arr = np.load(dense_rc)
            bm25_ranks_arr  = np.load(bm25_rc)
            
            for i in range(n_queries):
                if not valid_mask[i]:
                    continue
                # weighted RRF fusion (재실험코드 3과 동일: K_RRF=60)
                fused = weighted_rrf_fuse(
                    dense_ranks_arr[i], bm25_ranks_arr[i],
                    0.5, 0.5, K_RRF, out_topk=50
                )
                ranked_pids = [corpus_pids[int(d)] for d in fused]
                m = {"Hit@10": hit_at_k(ranked_pids, gold_pids[i], 10),
                     "MRR@10": mrr_at_k(ranked_pids, gold_pids[i], 10),
                     "Hit@50": hit_at_k(ranked_pids, gold_pids[i], 50)}
                per_hit10[i] = m["Hit@10"]
                per_mrr10[i] = m["MRR@10"]
                per_hit50[i] = m["Hit@50"]
        
        year_hits[model_id] = {"hit10": per_hit10, "hit50": per_hit50, "mrr10": per_mrr10}
        
        # 유효한 쿼리만으로 평균 계산하여 검증 출력
        valid_h10 = per_hit10[valid_mask]
        avg = valid_h10.mean() if len(valid_h10) > 0 else 0
        print(f"  {display_name:35s} Hit@10={avg:.4f}")
    
    all_query_hits[year] = year_hits

print("\nPhase 0 complete.")


# =============================================================================
# query-level 결과를 wide-format DataFrame으로 통합
# =============================================================================
print("\nBuilding unified query-level DataFrame...")

rows = []
for year in YEARS:
    meta = all_query_meta[year]
    hits = all_query_hits[year]
    
    for i in range(len(meta)):
        if not meta.iloc[i]["valid"]:
            continue
        row = {
            "year": year,
            "query_idx": i,
            "patent_id": meta.iloc[i]["patent_id"],
            "query_type": meta.iloc[i]["query_type"],
            "query_text": meta.iloc[i]["query_text"],
            "wipo_field": meta.iloc[i].get("wipo_field", None),
            "source_claims": meta.iloc[i]["source_claims"],
            "source_abstract": meta.iloc[i]["source_abstract"],
        }
        for model_id in hits:
            row[f"{model_id}_hit10"] = int(hits[model_id]["hit10"][i])
            row[f"{model_id}_hit50"] = int(hits[model_id]["hit50"][i])
            row[f"{model_id}_mrr10"] = float(hits[model_id]["mrr10"][i])
        rows.append(row)

df_all = pd.DataFrame(rows)
print(f"Unified DataFrame: {len(df_all)} queries × {len(df_all.columns)} columns")

# 저장
df_all.to_csv(os.path.join(OUT_DIR, "query_level_hits_all.csv"), index=False)
print(f"Saved: {os.path.join(OUT_DIR, 'query_level_hits_all.csv')}")


# #############################################################################
#
#  이슈 1: Query-source term overlap + Lexical shortcut 서브셋 분석
#
# #############################################################################
print("\n" + "=" * 70)
print("이슈 1: Query-Source Term Overlap & Lexical Shortcut Analysis")
print("=" * 70)

def tokenize_simple(text):
    """소문자 변환 후 알파벳/숫자 토큰 추출"""
    return re.findall(r'[a-z0-9]+', text.lower())

def jaccard_similarity(tokens_a, tokens_b):
    """두 토큰 집합의 Jaccard similarity"""
    set_a = set(tokens_a)
    set_b = set(tokens_b)
    if len(set_a) == 0 and len(set_b) == 0:
        return 0.0
    intersection = set_a & set_b
    union = set_a | set_b
    return len(intersection) / len(union) if len(union) > 0 else 0.0

def unigram_overlap_ratio(query_tokens, source_tokens):
    """쿼리 토큰 중 source에도 등장하는 비율"""
    if len(query_tokens) == 0:
        return 0.0
    source_set = set(source_tokens)
    overlap = sum(1 for t in query_tokens if t in source_set)
    return overlap / len(query_tokens)

# 1-A: Overlap 계산
print("\n--- 1-A: Computing query-source overlap ---")

overlaps = []
for _, row in df_all.iterrows():
    q_tokens = tokenize_simple(str(row["query_text"]))
    # source = claims + abstract 합산
    src_text = str(row["source_claims"]) + " " + str(row["source_abstract"])
    s_tokens = tokenize_simple(src_text)
    
    jac = jaccard_similarity(q_tokens, s_tokens)
    overlap_ratio = unigram_overlap_ratio(q_tokens, s_tokens)
    
    overlaps.append({
        "year": row["year"],
        "query_type": row["query_type"],
        "jaccard": jac,
        "overlap_ratio": overlap_ratio,
        "n_query_tokens": len(q_tokens),
    })

df_overlap = pd.DataFrame(overlaps)
df_all["jaccard"] = df_overlap["jaccard"].values
df_all["overlap_ratio"] = df_overlap["overlap_ratio"].values
df_all["n_query_tokens"] = df_overlap["n_query_tokens"].values

# query_type별 overlap 통계
print("\nQuery-source overlap by query type:")
overlap_stats = df_overlap.groupby("query_type").agg(
    jaccard_mean=("jaccard", "mean"),
    jaccard_median=("jaccard", "median"),
    jaccard_std=("jaccard", "std"),
    overlap_ratio_mean=("overlap_ratio", "mean"),
    overlap_ratio_median=("overlap_ratio", "median"),
    overlap_ratio_std=("overlap_ratio", "std"),
    n=("jaccard", "count"),
).round(4)
print(overlap_stats.to_string())
overlap_stats.to_csv(os.path.join(OUT_DIR, "issue1_overlap_by_query_type.csv"))
print(f"Saved: issue1_overlap_by_query_type.csv")

# 연도 × query_type별 overlap 통계
overlap_stats_year = df_overlap.groupby(["year", "query_type"]).agg(
    jaccard_mean=("jaccard", "mean"),
    jaccard_median=("jaccard", "median"),
    overlap_ratio_mean=("overlap_ratio", "mean"),
    overlap_ratio_median=("overlap_ratio", "median"),
    n=("jaccard", "count"),
).round(4)
overlap_stats_year.to_csv(os.path.join(OUT_DIR, "issue1_overlap_by_year_query_type.csv"))
print(f"Saved: issue1_overlap_by_year_query_type.csv")


# 1-B: Lexical shortcut 분석 — low/high overlap 서브셋별 성능 비교
print("\n--- 1-B: Lexical Shortcut Analysis (low vs high overlap subsets) ---")

# overlap_ratio 기준 4분위 분할
q25 = df_all["overlap_ratio"].quantile(0.25)
q75 = df_all["overlap_ratio"].quantile(0.75)
print(f"  Overlap ratio quartiles: Q25={q25:.4f}, Q75={q75:.4f}")

df_all["overlap_group"] = pd.cut(
    df_all["overlap_ratio"],
    bins=[-0.001, q25, q75, 1.001],
    labels=["low_overlap", "mid_overlap", "high_overlap"]
)

# 각 overlap 그룹별 모델 성능
hit10_cols = [c for c in df_all.columns if c.endswith("_hit10")]
model_ids_in_data = [c.replace("_hit10", "") for c in hit10_cols]

subset_results = []
for group in ["low_overlap", "mid_overlap", "high_overlap"]:
    subset = df_all[df_all["overlap_group"] == group]
    n_subset = len(subset)
    for mid in model_ids_in_data:
        col = f"{mid}_hit10"
        avg_hit10 = subset[col].mean()
        col_mrr = f"{mid}_mrr10"
        avg_mrr10 = subset[col_mrr].mean() if col_mrr in subset.columns else np.nan
        subset_results.append({
            "overlap_group": group,
            "n_queries": n_subset,
            "model_id": mid,
            "Hit@10": round(avg_hit10, 4),
            "MRR@10": round(avg_mrr10, 4),
        })

df_subset = pd.DataFrame(subset_results)
df_subset_pivot = df_subset.pivot_table(
    index="model_id", columns="overlap_group",
    values="Hit@10", aggfunc="first"
)[["low_overlap", "mid_overlap", "high_overlap"]]

# dense-bm25 gap 계산 (low overlap에서 hybrid의 가치가 더 큰지)
print("\nHit@10 by overlap group:")
print(df_subset_pivot.to_string())

df_subset.to_csv(os.path.join(OUT_DIR, "issue1_lexical_shortcut_by_overlap_group.csv"), index=False)
df_subset_pivot.to_csv(os.path.join(OUT_DIR, "issue1_lexical_shortcut_pivot.csv"))
print(f"Saved: issue1_lexical_shortcut_by_overlap_group.csv")
print(f"Saved: issue1_lexical_shortcut_pivot.csv")

# low overlap 서브셋에서 hybrid improvement 분석
print("\n--- Hybrid improvement in LOW overlap subset ---")
low_df = df_all[df_all["overlap_group"] == "low_overlap"]
pairs = [
    ("BGE_M3_512_CLAIMS_DENSE",    "BGE_M3_512_CLAIMS_HYBRID",    "BGE-M3 Claims"),
    ("BGE_M3_512_ABSTRACT_DENSE",  "BGE_M3_512_ABSTRACT_HYBRID",  "BGE-M3 Abstract"),
    ("PATENTSBERTA_CLAIMS_DENSE",  "PATENTSBERTA_CLAIMS_HYBRID",  "PatentSBERTa Claims"),
    ("PATENTSBERTA_ABSTRACT_DENSE","PATENTSBERTA_ABSTRACT_HYBRID", "PatentSBERTa Abstract"),
]
for dense_id, hybrid_id, label in pairs:
    d_col = f"{dense_id}_hit10"
    h_col = f"{hybrid_id}_hit10"
    if d_col in low_df.columns and h_col in low_df.columns:
        d_mean = low_df[d_col].mean()
        h_mean = low_df[h_col].mean()
        delta = h_mean - d_mean
        print(f"  {label:25s}: Dense={d_mean:.4f}, Hybrid={h_mean:.4f}, Δ={delta:+.4f}")


# #############################################################################
#
#  이슈 2: Paired Bootstrap 통계 검정
#
# #############################################################################
print("\n" + "=" * 70)
print("이슈 2: Paired Bootstrap Statistical Tests")
print("=" * 70)

def paired_bootstrap_test(hits_a, hits_b, n_bootstrap=1000, seed=42):
    """
    두 모델의 query-level hit@10 (0/1)에 대한 paired bootstrap test.
    
    Returns:
        delta_mean: 평균 차이 (A - B)
        ci_lo, ci_hi: 95% CI of the difference
        p_value: 양측 p-value (H0: delta = 0)
    """
    rng = np.random.RandomState(seed)
    n = len(hits_a)
    observed_delta = hits_a.mean() - hits_b.mean()
    
    deltas = np.zeros(n_bootstrap)
    for b in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        deltas[b] = hits_a[idx].mean() - hits_b[idx].mean()
    
    ci_lo = np.percentile(deltas, 2.5)
    ci_hi = np.percentile(deltas, 97.5)
    
    # p-value: 부호 반전 비율 (양측)
    p_value = np.mean(deltas * np.sign(observed_delta) <= 0) if observed_delta != 0 else 1.0
    # 더 정확한 양측 p-value
    centered = deltas - deltas.mean()
    p_value = np.mean(np.abs(centered) >= np.abs(observed_delta))
    
    return observed_delta, ci_lo, ci_hi, p_value

N_BOOTSTRAP = 1000

# 핵심 비교 쌍 정의
comparison_pairs = [
    # (model_A, model_B, 설명)
    # RQ2: BGE-M3 vs PatentSBERTa (hybrid, 동일 field)
    ("BGE_M3_512_CLAIMS_HYBRID",   "PATENTSBERTA_CLAIMS_HYBRID",   "BGE-M3 vs PatentSBERTa (Claims Hybrid)"),
    ("BGE_M3_512_ABSTRACT_HYBRID", "PATENTSBERTA_ABSTRACT_HYBRID", "BGE-M3 vs PatentSBERTa (Abstract Hybrid)"),
    
    # Hybrid vs Dense (각 모델별)
    ("BGE_M3_512_CLAIMS_HYBRID",    "BGE_M3_512_CLAIMS_DENSE",     "BGE-M3 Claims: Hybrid vs Dense"),
    ("BGE_M3_512_ABSTRACT_HYBRID",  "BGE_M3_512_ABSTRACT_DENSE",   "BGE-M3 Abstract: Hybrid vs Dense"),
    ("PATENTSBERTA_CLAIMS_HYBRID",  "PATENTSBERTA_CLAIMS_DENSE",   "PatentSBERTa Claims: Hybrid vs Dense"),
    ("PATENTSBERTA_ABSTRACT_HYBRID","PATENTSBERTA_ABSTRACT_DENSE",  "PatentSBERTa Abstract: Hybrid vs Dense"),
    
    # BM25 vs Dense
    ("BM25_CLAIMS",                 "BGE_M3_512_CLAIMS_DENSE",     "BM25_Claims vs BGE-M3 Claims Dense"),
    ("BM25_CLAIMS",                 "PATENTSBERTA_CLAIMS_DENSE",   "BM25_Claims vs PatentSBERTa Claims Dense"),
    
    # Top hybrid vs BM25
    ("BGE_M3_512_CLAIMS_HYBRID",    "BM25_CLAIMS",                 "BGE-M3 Claims Hybrid vs BM25_Claims"),
]

bootstrap_results = []
for model_a, model_b, description in comparison_pairs:
    col_a = f"{model_a}_hit10"
    col_b = f"{model_b}_hit10"
    
    if col_a not in df_all.columns or col_b not in df_all.columns:
        print(f"  [SKIP] {description}: columns not found")
        continue
    
    hits_a = df_all[col_a].values.astype(float)
    hits_b = df_all[col_b].values.astype(float)
    
    delta, ci_lo, ci_hi, p_val = paired_bootstrap_test(hits_a, hits_b, N_BOOTSTRAP)
    
    result = {
        "model_A": model_a,
        "model_B": model_b,
        "description": description,
        "mean_A": round(hits_a.mean(), 4),
        "mean_B": round(hits_b.mean(), 4),
        "delta": round(delta, 4),
        "CI_lo": round(ci_lo, 4),
        "CI_hi": round(ci_hi, 4),
        "p_value": round(p_val, 4),
        "significant_005": "Yes" if p_val < 0.05 else "No",
        "n_queries": len(hits_a),
    }
    bootstrap_results.append(result)
    
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
    print(f"  {description}")
    print(f"    A={hits_a.mean():.4f}, B={hits_b.mean():.4f}, Δ={delta:+.4f} "
          f"[{ci_lo:+.4f}, {ci_hi:+.4f}], p={p_val:.4f} {sig}")

df_bootstrap = pd.DataFrame(bootstrap_results)
df_bootstrap.to_csv(os.path.join(OUT_DIR, "issue2_paired_bootstrap_results.csv"), index=False)
print(f"\nSaved: issue2_paired_bootstrap_results.csv")

# query_type별 paired bootstrap (주요 비교만)
print("\n--- Query-type specific bootstrap tests ---")
key_pairs = [
    ("BGE_M3_512_CLAIMS_HYBRID", "PATENTSBERTA_CLAIMS_HYBRID", "BGE-M3 vs PatentSBERTa Claims Hybrid"),
]
type_bootstrap_results = []
for model_a, model_b, desc in key_pairs:
    col_a = f"{model_a}_hit10"
    col_b = f"{model_b}_hit10"
    for qt in QUERY_TYPES:
        subset = df_all[df_all["query_type"] == qt]
        ha = subset[col_a].values.astype(float)
        hb = subset[col_b].values.astype(float)
        delta, ci_lo, ci_hi, p_val = paired_bootstrap_test(ha, hb, N_BOOTSTRAP)
        type_bootstrap_results.append({
            "comparison": desc,
            "query_type": qt,
            "mean_A": round(ha.mean(), 4),
            "mean_B": round(hb.mean(), 4),
            "delta": round(delta, 4),
            "CI_lo": round(ci_lo, 4),
            "CI_hi": round(ci_hi, 4),
            "p_value": round(p_val, 4),
            "n": len(ha),
        })
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
        print(f"  [{qt}] Δ={delta:+.4f} [{ci_lo:+.4f},{ci_hi:+.4f}] p={p_val:.4f} {sig}")

df_type_bootstrap = pd.DataFrame(type_bootstrap_results)
df_type_bootstrap.to_csv(os.path.join(OUT_DIR, "issue2_paired_bootstrap_by_query_type.csv"), index=False)
print(f"Saved: issue2_paired_bootstrap_by_query_type.csv")


# #############################################################################
#
#  이슈 3: PatentSBERTa 2-Way Grid Search (Equal-Weight Bias 검증)
#
# #############################################################################
print("\n" + "=" * 70)
print("이슈 3: PatentSBERTa 2-Way Grid Search")
print("=" * 70)
print("  0.05 간격으로 21개 weight 조합 × 2 fields × 3 years × 3 query types")

WEIGHT_GRID = np.arange(0.0, 1.05, 0.05).round(2)  # 0.00, 0.05, ..., 1.00

def grid_search_2way(years, query_types, model_encoder, max_length, field):
    """
    주어진 모델-필드에 대해 2-way grid search 수행.
    각 weight에서 전체 pooled Hit@10, WorstYear Hit@10 계산.
    patent_id 문자열 비교 방식 (재실험코드 3과 동일).
    """
    # 코퍼스를 미리 로드 (21개 weight 반복 중 재로드 방지)
    corpus_cache = {}
    for year in years:
        c = load_corpus(year)
        c = c.drop_duplicates(subset="patent_id", keep="first").reset_index(drop=True)
        corpus_cache[year] = c["patent_id"].tolist()
    
    results = []
    
    for w_dense in WEIGHT_GRID:
        w_bm25 = round(1.0 - w_dense, 2)
        
        year_means = {}
        
        for year in years:
            meta = all_query_meta[year]
            cpids = corpus_cache[year]
            gold_pids_y = meta["patent_id"].astype(str).tolist()
            pid_set = set(cpids)
            vmask = np.array([pid in pid_set for pid in gold_pids_y], dtype=bool)
            
            dense_rc = _rank_cache_path(year, "dense", field, model_encoder, max_length)
            bm25_rc  = _rank_cache_path(year, "bm25", field)
            
            if not os.path.exists(dense_rc) or not os.path.exists(bm25_rc):
                continue
            
            dense_ranks = np.load(dense_rc)
            bm25_ranks  = np.load(bm25_rc)
            
            n_q = len(meta)
            per_h10 = np.zeros(n_q)
            per_m10 = np.zeros(n_q)
            per_h50 = np.zeros(n_q)
            
            for i in range(n_q):
                if not vmask[i]:
                    continue
                fused = weighted_rrf_fuse(
                    dense_ranks[i], bm25_ranks[i],
                    w_dense, w_bm25, K_RRF, out_topk=50
                )
                rpids = [cpids[int(d)] for d in fused]
                per_h10[i] = hit_at_k(rpids, gold_pids_y[i], 10)
                per_m10[i] = mrr_at_k(rpids, gold_pids_y[i], 10)
                per_h50[i] = hit_at_k(rpids, gold_pids_y[i], 50)
            
            # query_type별 평균 → macro average
            type_means_h10 = []
            type_means_m10 = []
            type_means_h50 = []
            for qt in query_types:
                qt_mask = (meta["query_type"] == qt).values & vmask
                if qt_mask.sum() > 0:
                    type_means_h10.append(per_h10[qt_mask].mean())
                    type_means_m10.append(per_m10[qt_mask].mean())
                    type_means_h50.append(per_h50[qt_mask].mean())
            
            macro_h10 = np.mean(type_means_h10)
            macro_m10 = np.mean(type_means_m10)
            macro_h50 = np.mean(type_means_h50)
            year_means[year] = (macro_h10, macro_m10, macro_h50)
        
        if len(year_means) == 0:
            continue
        
        overall_h10 = np.mean([v[0] for v in year_means.values()])
        overall_m10 = np.mean([v[1] for v in year_means.values()])
        overall_h50 = np.mean([v[2] for v in year_means.values()])
        worst_h10 = min(v[0] for v in year_means.values())
        worst_m10 = min(v[1] for v in year_means.values())
        worst_h50 = min(v[2] for v in year_means.values())
        
        results.append({
            "weight_dense": w_dense,
            "weight_bm25": w_bm25,
            "Overall_Hit@10": round(overall_h10, 6),
            "Overall_MRR@10": round(overall_m10, 6),
            "Overall_Hit@50": round(overall_h50, 6),
            "WorstYear_Hit@10": round(worst_h10, 6),
            "WorstYear_MRR@10": round(worst_m10, 6),
            "WorstYear_Hit@50": round(worst_h50, 6),
        })
    
    return pd.DataFrame(results)

# PatentSBERTa Claims Hybrid grid search
print("\n--- PatentSBERTa Claims ---")
df_ps_claims = grid_search_2way(
    YEARS, QUERY_TYPES,
    model_encoder="AI-Growth-Lab/PatentSBERTa", max_length=512, field="claims"
)
df_ps_claims = df_ps_claims.sort_values("Overall_Hit@10", ascending=False)
print(f"  Best weight: dense={df_ps_claims.iloc[0]['weight_dense']}, "
      f"bm25={df_ps_claims.iloc[0]['weight_bm25']}, "
      f"Hit@10={df_ps_claims.iloc[0]['Overall_Hit@10']:.4f}")
df_ps_claims.to_csv(os.path.join(OUT_DIR, "issue3_gridsearch_patentsberta_claims.csv"), index=False)

# PatentSBERTa Abstract Hybrid grid search
print("\n--- PatentSBERTa Abstract ---")
df_ps_abstract = grid_search_2way(
    YEARS, QUERY_TYPES,
    model_encoder="AI-Growth-Lab/PatentSBERTa", max_length=512, field="abstract"
)
df_ps_abstract = df_ps_abstract.sort_values("Overall_Hit@10", ascending=False)
print(f"  Best weight: dense={df_ps_abstract.iloc[0]['weight_dense']}, "
      f"bm25={df_ps_abstract.iloc[0]['weight_bm25']}, "
      f"Hit@10={df_ps_abstract.iloc[0]['Overall_Hit@10']:.4f}")
df_ps_abstract.to_csv(os.path.join(OUT_DIR, "issue3_gridsearch_patentsberta_abstract.csv"), index=False)

# BGE-M3 Claims Hybrid grid search (비교용)
print("\n--- BGE-M3 Claims ---")
df_bge_claims = grid_search_2way(
    YEARS, QUERY_TYPES,
    model_encoder="BAAI/bge-m3", max_length=512, field="claims"
)
df_bge_claims = df_bge_claims.sort_values("Overall_Hit@10", ascending=False)
print(f"  Best weight: dense={df_bge_claims.iloc[0]['weight_dense']}, "
      f"bm25={df_bge_claims.iloc[0]['weight_bm25']}, "
      f"Hit@10={df_bge_claims.iloc[0]['Overall_Hit@10']:.4f}")
df_bge_claims.to_csv(os.path.join(OUT_DIR, "issue3_gridsearch_bgem3_claims.csv"), index=False)

# Per-model optimal weight 비교 테이블
print("\n--- Per-Model Optimal Weight Comparison ---")
summary_rows = []
for label, df_grid, field in [
    ("PatentSBERTa_Claims",  df_ps_claims,  "claims"),
    ("PatentSBERTa_Abstract",df_ps_abstract,"abstract"),
    ("BGE-M3_Claims",        df_bge_claims, "claims"),
]:
    best = df_grid.iloc[0]
    # 0.5/0.5에서의 성능도 찾기
    eq_row = df_grid[df_grid["weight_dense"] == 0.5]
    eq_hit10 = eq_row.iloc[0]["Overall_Hit@10"] if len(eq_row) > 0 else np.nan
    
    summary_rows.append({
        "model_field": label,
        "optimal_w_dense": best["weight_dense"],
        "optimal_w_bm25": best["weight_bm25"],
        "optimal_Hit@10": best["Overall_Hit@10"],
        "equal_weight_Hit@10": round(eq_hit10, 6),
        "delta_optimal_vs_equal": round(best["Overall_Hit@10"] - eq_hit10, 6),
    })

# BGE-M3 Abstract (기존 데이터에서)
bge_abs_path = os.path.join(RESULTS_DIR_NB3, "..", "notebook03_eval_v2",
                            "grid_summary_2way_abstract_denseL512_P200_D200.csv")
# 대안 경로
for candidate in [
    os.path.join(OUT_DIR, "..", "notebook03_eval_v2", "grid_summary_2way_abstract_denseL512_P200_D200.csv"),
    "grid_summary_2way_abstract_denseL512_P200_D200.csv",
]:
    if os.path.exists(candidate):
        bge_abs_path = candidate
        break

df_summary_optimal = pd.DataFrame(summary_rows)
print(df_summary_optimal.to_string(index=False))
df_summary_optimal.to_csv(os.path.join(OUT_DIR, "issue3_per_model_optimal_weights.csv"), index=False)
print(f"\nSaved: issue3_per_model_optimal_weights.csv")

# Per-model optimal에서의 모델 간 비교
print("\n--- At each model's optimal weight, does BGE-M3 > PatentSBERTa hold? ---")
for field_label in ["claims"]:
    bge_best = df_bge_claims.iloc[0]["Overall_Hit@10"]
    ps_best  = df_ps_claims.iloc[0]["Overall_Hit@10"]
    print(f"  {field_label}: BGE-M3 optimal={bge_best:.4f}, PatentSBERTa optimal={ps_best:.4f}, "
          f"Δ={bge_best - ps_best:+.4f}")


# #############################################################################
#
#  이슈 4: WIPO 5대 섹터별 Difficulty Decomposition
#
# #############################################################################
print("\n" + "=" * 70)
print("이슈 4: WIPO Sector-Level Difficulty Decomposition")
print("=" * 70)

# WIPO technology field → 5대 섹터 매핑
# Reference: WIPO Technology Classification (35 fields → 5 sectors)
WIPO_SECTOR_MAP = {
    # Sector 1: Electrical engineering (fields 1-8)
    1: "Electrical_engineering",  # Electrical machinery, apparatus, energy
    2: "Electrical_engineering",  # Audio-visual technology
    3: "Electrical_engineering",  # Telecommunications
    4: "Electrical_engineering",  # Digital communication
    5: "Electrical_engineering",  # Basic communication processes
    6: "Electrical_engineering",  # Computer technology
    7: "Electrical_engineering",  # IT methods for management
    8: "Electrical_engineering",  # Semiconductors
    
    # Sector 2: Instruments (fields 9-13)
    9:  "Instruments",   # Optics
    10: "Instruments",   # Measurement
    11: "Instruments",   # Analysis of biological materials
    12: "Instruments",   # Control
    13: "Instruments",   # Medical technology
    
    # Sector 3: Chemistry (fields 14-24)
    14: "Chemistry",     # Organic fine chemistry
    15: "Chemistry",     # Biotechnology
    16: "Chemistry",     # Pharmaceuticals
    17: "Chemistry",     # Macromolecular chemistry, polymers
    18: "Chemistry",     # Food chemistry
    19: "Chemistry",     # Basic materials chemistry
    20: "Chemistry",     # Materials, metallurgy
    21: "Chemistry",     # Surface technology, coating
    22: "Chemistry",     # Micro-structure and nano-technology
    23: "Chemistry",     # Chemical engineering
    24: "Chemistry",     # Environmental technology
    
    # Sector 4: Mechanical engineering (fields 25-32)
    25: "Mechanical_engineering",  # Handling
    26: "Mechanical_engineering",  # Machine tools
    27: "Mechanical_engineering",  # Engines, pumps, turbines
    28: "Mechanical_engineering",  # Textile and paper machines
    29: "Mechanical_engineering",  # Other special machines
    30: "Mechanical_engineering",  # Thermal processes and apparatus
    31: "Mechanical_engineering",  # Mechanical elements
    32: "Mechanical_engineering",  # Transport
    
    # Sector 5: Other fields (fields 33-35)
    33: "Other_fields",  # Furniture, games
    34: "Other_fields",  # Other consumer goods
    35: "Other_fields",  # Civil engineering
}

# wipo_field를 숫자로 변환하고 섹터 매핑
if "wipo_field" in df_all.columns:
    df_all["wipo_field_num"] = pd.to_numeric(df_all["wipo_field"], errors="coerce")
    df_all["wipo_sector"] = df_all["wipo_field_num"].map(WIPO_SECTOR_MAP)
    
    n_mapped = df_all["wipo_sector"].notna().sum()
    print(f"  WIPO sector mapped: {n_mapped} / {len(df_all)} queries")
    print(f"  Sector distribution:")
    print(df_all["wipo_sector"].value_counts().to_string())
    
    # 섹터별 모델 성능
    sector_results = []
    for sector in sorted(df_all["wipo_sector"].dropna().unique()):
        subset = df_all[df_all["wipo_sector"] == sector]
        n_sector = len(subset)
        
        for mid in model_ids_in_data:
            col_h10 = f"{mid}_hit10"
            col_m10 = f"{mid}_mrr10"
            
            avg_h10 = subset[col_h10].mean()
            avg_m10 = subset[col_m10].mean() if col_m10 in subset.columns else np.nan
            
            sector_results.append({
                "wipo_sector": sector,
                "n_queries": n_sector,
                "model_id": mid,
                "Hit@10": round(avg_h10, 4),
                "MRR@10": round(avg_m10, 4),
            })
    
    df_sector = pd.DataFrame(sector_results)
    df_sector.to_csv(os.path.join(OUT_DIR, "issue4_wipo_sector_performance.csv"), index=False)
    print(f"\nSaved: issue4_wipo_sector_performance.csv")
    
    # 피봇 테이블: 모델 × 섹터 Hit@10
    pivot = df_sector.pivot_table(
        index="model_id", columns="wipo_sector",
        values="Hit@10", aggfunc="first"
    )
    print("\nHit@10 by WIPO sector:")
    print(pivot.round(3).to_string())
    pivot.to_csv(os.path.join(OUT_DIR, "issue4_wipo_sector_pivot.csv"))
    print(f"Saved: issue4_wipo_sector_pivot.csv")
    
    # 섹터별 hybrid improvement 분석 (sparse vs dense balance가 다른지)
    print("\n--- Hybrid improvement by WIPO sector ---")
    sector_hybrid_improvement = []
    for sector in sorted(df_all["wipo_sector"].dropna().unique()):
        subset = df_all[df_all["wipo_sector"] == sector]
        
        for dense_id, hybrid_id, label in pairs:
            d_col = f"{dense_id}_hit10"
            h_col = f"{hybrid_id}_hit10"
            b_col = "BM25_CLAIMS_hit10"
            
            if d_col in subset.columns and h_col in subset.columns:
                d_mean = subset[d_col].mean()
                h_mean = subset[h_col].mean()
                b_mean = subset[b_col].mean() if b_col in subset.columns else np.nan
                
                sector_hybrid_improvement.append({
                    "wipo_sector": sector,
                    "comparison": label,
                    "dense_Hit@10": round(d_mean, 4),
                    "hybrid_Hit@10": round(h_mean, 4),
                    "bm25_claims_Hit@10": round(b_mean, 4),
                    "delta_hybrid_dense": round(h_mean - d_mean, 4),
                    "n_queries": len(subset),
                })
    
    df_sector_hybrid = pd.DataFrame(sector_hybrid_improvement)
    df_sector_hybrid.to_csv(os.path.join(OUT_DIR, "issue4_wipo_sector_hybrid_improvement.csv"), index=False)
    print(f"Saved: issue4_wipo_sector_hybrid_improvement.csv")
    
    # 섹터별 hybrid improvement 요약 출력
    for sector in sorted(df_all["wipo_sector"].dropna().unique()):
        s_data = df_sector_hybrid[df_sector_hybrid["wipo_sector"] == sector]
        bge_claims = s_data[s_data["comparison"] == "BGE-M3 Claims"]
        if len(bge_claims) > 0:
            row = bge_claims.iloc[0]
            print(f"  {sector:25s}: Dense={row['dense_Hit@10']:.3f}, "
                  f"Hybrid={row['hybrid_Hit@10']:.3f}, "
                  f"BM25={row['bm25_claims_Hit@10']:.3f}, "
                  f"Δ(H-D)={row['delta_hybrid_dense']:+.3f}")

else:
    print("  [WARN] wipo_field column not found in query metadata. Skipping sector analysis.")


# #############################################################################
#
#  논문용 Figure & Table 생성
#
# #############################################################################
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

FIGURES_DIR = os.path.join(OUT_DIR, "figures")
TABLES_DIR  = os.path.join(OUT_DIR, "tables")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR,  exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 이슈 1  Figure: Overlap group별 Dense vs Hybrid Hit@10 (grouped bar)
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Generating Issue 1 figures & tables ---")

# Figure 1-A: 4개 모델쌍의 hybrid improvement를 overlap group별로 비교
fig, ax = plt.subplots(figsize=(6.5, 3.5))

groups = ["low_overlap", "mid_overlap", "high_overlap"]
group_labels = ["Low overlap\n(Q1)", "Mid overlap\n(Q2–Q3)", "High overlap\n(Q4)"]
model_pairs_plot = [
    ("BGE_M3_512_CLAIMS_DENSE",     "BGE_M3_512_CLAIMS_HYBRID",     "BGE-M3 Claims"),
    ("BGE_M3_512_ABSTRACT_DENSE",   "BGE_M3_512_ABSTRACT_HYBRID",   "BGE-M3 Abstract"),
    ("PATENTSBERTA_CLAIMS_DENSE",   "PATENTSBERTA_CLAIMS_HYBRID",   "PatentSBERTa Claims"),
    ("PATENTSBERTA_ABSTRACT_DENSE", "PATENTSBERTA_ABSTRACT_HYBRID", "PatentSBERTa Abstract"),
]
colors_dense  = ["#4e79a7", "#76b7b2", "#e15759", "#f28e2b"]
colors_hybrid = ["#1b4f72", "#117a65", "#922b21", "#b9770e"]

n_pairs = len(model_pairs_plot)
n_groups = len(groups)
bar_w = 0.09
x_base = np.arange(n_groups)

for p_idx, (dense_id, hybrid_id, label) in enumerate(model_pairs_plot):
    d_vals = []
    h_vals = []
    for g in groups:
        sub = df_all[df_all["overlap_group"] == g]
        d_vals.append(sub[f"{dense_id}_hit10"].mean())
        h_vals.append(sub[f"{hybrid_id}_hit10"].mean())
    
    offset = (p_idx - n_pairs / 2 + 0.5) * bar_w * 2.2
    ax.bar(x_base + offset - bar_w * 0.55, d_vals, bar_w, color=colors_dense[p_idx],
           alpha=0.55, label=f"{label} Dense" if p_idx == 0 else None)
    ax.bar(x_base + offset + bar_w * 0.55, h_vals, bar_w, color=colors_hybrid[p_idx],
           alpha=0.85, label=f"{label} Hybrid" if p_idx == 0 else None)
    
    # delta 표시
    for g_idx in range(n_groups):
        delta = h_vals[g_idx] - d_vals[g_idx]
        ax.annotate(f"+{delta:.1%}" if delta > 0 else f"{delta:.1%}",
                    xy=(x_base[g_idx] + offset + bar_w * 0.55, h_vals[g_idx]),
                    fontsize=5.5, ha="center", va="bottom", color=colors_hybrid[p_idx])

ax.set_xticks(x_base)
ax.set_xticklabels(group_labels)
ax.set_ylabel("Hit@10")
ax.set_ylim(0.65, 1.0)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

# custom legend: Dense (light) / Hybrid (dark) + model labels
from matplotlib.patches import Patch
legend_elements = []
for p_idx, (_, _, label) in enumerate(model_pairs_plot):
    legend_elements.append(Patch(facecolor=colors_dense[p_idx], alpha=0.55, label=f"{label} Dense"))
    legend_elements.append(Patch(facecolor=colors_hybrid[p_idx], alpha=0.85, label=f"{label} Hybrid"))
ax.legend(handles=legend_elements, ncol=2, loc="lower right", framealpha=0.9,
          fontsize=6.5, handlelength=1.2, handletextpad=0.4, columnspacing=0.8)

ax.set_title("Hybrid Improvement by Query–Source Lexical Overlap Group")
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "issue1_hybrid_improvement_by_overlap.png"))
fig.savefig(os.path.join(FIGURES_DIR, "issue1_hybrid_improvement_by_overlap.pdf"))
plt.close(fig)
print("  Saved: issue1_hybrid_improvement_by_overlap.png/pdf")

# Figure 1-B: overlap distribution by query type (box plot)
fig, ax = plt.subplots(figsize=(4.5, 3.0))
qt_data = [df_all[df_all["query_type"] == qt]["overlap_ratio"].values for qt in QUERY_TYPES]
bp = ax.boxplot(qt_data, labels=["Summary", "Keyword", "Function"],
                patch_artist=True, widths=0.5,
                medianprops=dict(color="black", linewidth=1.5))
box_colors = ["#aec7e8", "#ffbb78", "#98df8a"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Unigram Overlap Ratio\n(query tokens in source)")
ax.set_title("Query–Source Lexical Overlap by Query Type")
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "issue1_overlap_distribution_by_type.png"))
fig.savefig(os.path.join(FIGURES_DIR, "issue1_overlap_distribution_by_type.pdf"))
plt.close(fig)
print("  Saved: issue1_overlap_distribution_by_type.png/pdf")

# Table 1: Overlap statistics (논문 본문용 LaTeX-friendly)
tbl1_lines = []
tbl1_lines.append("Query Type | N | Jaccard (mean±std) | Overlap Ratio (mean±std)")
tbl1_lines.append("-" * 75)
for qt in QUERY_TYPES:
    sub = df_overlap[df_overlap["query_type"] == qt]
    jm, js = sub["jaccard"].mean(), sub["jaccard"].std()
    om, os_ = sub["overlap_ratio"].mean(), sub["overlap_ratio"].std()
    tbl1_lines.append(f"{qt:10s} | {len(sub):5d} | {jm:.4f} ± {js:.4f}       | {om:.4f} ± {os_:.4f}")
tbl1_text = "\n".join(tbl1_lines)
with open(os.path.join(TABLES_DIR, "issue1_overlap_statistics.txt"), "w") as f:
    f.write(tbl1_text)
print("  Saved: issue1_overlap_statistics.txt")
print(tbl1_text)

# Table 1-B: Lexical shortcut — key models × overlap groups
tbl1b_models = [
    ("BGE_M3_512_CLAIMS_HYBRID",   "BGE-M3 Claims Hybrid"),
    ("PATENTSBERTA_CLAIMS_HYBRID", "PatentSBERTa Claims Hybrid"),
    ("BGE_M3_512_CLAIMS_DENSE",    "BGE-M3 Claims Dense"),
    ("PATENTSBERTA_CLAIMS_DENSE",  "PatentSBERTa Claims Dense"),
    ("BM25_CLAIMS",                "BM25 Claims"),
]
tbl1b_lines = []
tbl1b_lines.append(f"{'Model':<30s} | {'Low':>8s} | {'Mid':>8s} | {'High':>8s} | {'Δ(H-L)':>8s}")
tbl1b_lines.append("-" * 75)
for mid, display in tbl1b_models:
    col = f"{mid}_hit10"
    if col not in df_all.columns:
        continue
    vals = {}
    for g in groups:
        sub = df_all[df_all["overlap_group"] == g]
        vals[g] = sub[col].mean()
    delta = vals["high_overlap"] - vals["low_overlap"]
    tbl1b_lines.append(f"{display:<30s} | {vals['low_overlap']:8.4f} | "
                       f"{vals['mid_overlap']:8.4f} | {vals['high_overlap']:8.4f} | "
                       f"{delta:+8.4f}")
tbl1b_text = "\n".join(tbl1b_lines)
with open(os.path.join(TABLES_DIR, "issue1_lexical_shortcut_table.txt"), "w") as f:
    f.write(tbl1b_text)
print("  Saved: issue1_lexical_shortcut_table.txt")
print(tbl1b_text)


# ─────────────────────────────────────────────────────────────────────────────
# 이슈 2  Table: Paired bootstrap results (forest-plot style table)
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Generating Issue 2 tables ---")

# 논문 본문 Table: 핵심 비교 쌍
tbl2_lines = []
tbl2_lines.append(f"{'Comparison':<45s} | {'A':>7s} | {'B':>7s} | {'Δ':>7s} | {'95% CI':>17s} | {'p':>7s} | {'Sig.':>4s}")
tbl2_lines.append("-" * 110)
for _, row in df_bootstrap.iterrows():
    sig = "***" if row["p_value"] < 0.001 else "**" if row["p_value"] < 0.01 else "*" if row["p_value"] < 0.05 else "n.s."
    ci_str = f"[{row['CI_lo']:+.4f}, {row['CI_hi']:+.4f}]"
    tbl2_lines.append(f"{row['description']:<45s} | {row['mean_A']:7.4f} | {row['mean_B']:7.4f} | "
                      f"{row['delta']:+7.4f} | {ci_str:>17s} | {row['p_value']:7.4f} | {sig:>4s}")
tbl2_text = "\n".join(tbl2_lines)
with open(os.path.join(TABLES_DIR, "issue2_bootstrap_table.txt"), "w") as f:
    f.write(tbl2_text)
print("  Saved: issue2_bootstrap_table.txt")
print(tbl2_text)

# Figure 2: Forest plot of bootstrap CIs
fig, ax = plt.subplots(figsize=(6.5, 4.0))
n_comp = len(df_bootstrap)
y_pos = np.arange(n_comp)[::-1]

for i, (_, row) in enumerate(df_bootstrap.iterrows()):
    color = "#2171b5" if row["delta"] > 0 else "#cb181d"
    ax.plot([row["CI_lo"], row["CI_hi"]], [y_pos[i], y_pos[i]], color=color,
            linewidth=2.0, solid_capstyle="round")
    ax.plot(row["delta"], y_pos[i], "o", color=color, markersize=6, zorder=5)

ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels([r["description"] for _, r in df_bootstrap.iterrows()], fontsize=7)
ax.set_xlabel("Δ Hit@10 (Model A − Model B)")
ax.set_title("Paired Bootstrap 95% CI for Hit@10 Differences")
ax.grid(axis="x", alpha=0.3, linewidth=0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "issue2_forest_plot_bootstrap.png"))
fig.savefig(os.path.join(FIGURES_DIR, "issue2_forest_plot_bootstrap.pdf"))
plt.close(fig)
print("  Saved: issue2_forest_plot_bootstrap.png/pdf")


# ─────────────────────────────────────────────────────────────────────────────
# 이슈 3  Figure: 2-Way weight curves (PatentSBERTa vs BGE-M3)
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Generating Issue 3 figures & tables ---")

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.2), sharey=True)

# Left panel: Claims
ax = axes[0]
ax.plot(df_bge_claims["weight_dense"], df_bge_claims["Overall_Hit@10"],
        "o-", color="#2171b5", markersize=3, linewidth=1.5, label="BGE-M3 Claims")
ax.plot(df_ps_claims["weight_dense"], df_ps_claims["Overall_Hit@10"],
        "s-", color="#cb181d", markersize=3, linewidth=1.5, label="PatentSBERTa Claims")

# optimal markers
bge_best_c = df_bge_claims.iloc[0]
ps_best_c  = df_ps_claims.iloc[0]
ax.plot(bge_best_c["weight_dense"], bge_best_c["Overall_Hit@10"],
        "*", color="#2171b5", markersize=12, zorder=5)
ax.plot(ps_best_c["weight_dense"], ps_best_c["Overall_Hit@10"],
        "*", color="#cb181d", markersize=12, zorder=5)
ax.annotate(f'opt={bge_best_c["weight_dense"]:.2f}\n{bge_best_c["Overall_Hit@10"]:.3f}',
            xy=(bge_best_c["weight_dense"], bge_best_c["Overall_Hit@10"]),
            fontsize=6, color="#2171b5", ha="center", va="bottom",
            xytext=(0, 8), textcoords="offset points")
ax.annotate(f'opt={ps_best_c["weight_dense"]:.2f}\n{ps_best_c["Overall_Hit@10"]:.3f}',
            xy=(ps_best_c["weight_dense"], ps_best_c["Overall_Hit@10"]),
            fontsize=6, color="#cb181d", ha="center", va="bottom",
            xytext=(0, 8), textcoords="offset points")

ax.set_xlabel("Dense Weight (w_dense)")
ax.set_ylabel("Overall Hit@10")
ax.set_title("Claims Field")
ax.legend(fontsize=7, loc="lower center")
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_xlim(-0.02, 1.02)

# Right panel: Abstract
ax = axes[1]
ax.plot(df_ps_abstract["weight_dense"], df_ps_abstract["Overall_Hit@10"],
        "s-", color="#cb181d", markersize=3, linewidth=1.5, label="PatentSBERTa Abstract")

# BGE-M3 Abstract: 기존 grid search 결과 로드 시도
bge_abs_loaded = False
for candidate_path in [
    os.path.join(RESULTS_DIR_NB3, "grid_summary_2way_abstract_denseL512_P200_D200.csv"),
    os.path.join(BASE_DIR, "results", "notebook03_eval_v2",
                 "grid_summary_2way_abstract_denseL512_P200_D200.csv"),
    "grid_summary_2way_abstract_denseL512_P200_D200.csv",
]:
    if os.path.exists(candidate_path):
        df_bge_abs_existing = pd.read_csv(candidate_path)
        ax.plot(df_bge_abs_existing["weight_dense"], df_bge_abs_existing["Overall_Hit@10"],
                "o-", color="#2171b5", markersize=3, linewidth=1.5, label="BGE-M3 Abstract")
        bge_abs_best = df_bge_abs_existing.sort_values("Overall_Hit@10", ascending=False).iloc[0]
        ax.plot(bge_abs_best["weight_dense"], bge_abs_best["Overall_Hit@10"],
                "*", color="#2171b5", markersize=12, zorder=5)
        bge_abs_loaded = True
        break

if not bge_abs_loaded:
    # BGE-M3 Abstract grid search도 코드 내에서 실행
    print("  [INFO] Running BGE-M3 Abstract 2-way grid search...")
    df_bge_abstract = grid_search_2way(
        YEARS, QUERY_TYPES,
        model_encoder="BAAI/bge-m3", max_length=512, field="abstract"
    )
    df_bge_abstract = df_bge_abstract.sort_values("Overall_Hit@10", ascending=False)
    df_bge_abstract.to_csv(os.path.join(OUT_DIR, "issue3_gridsearch_bgem3_abstract.csv"), index=False)
    ax.plot(df_bge_abstract["weight_dense"], df_bge_abstract["Overall_Hit@10"],
            "o-", color="#2171b5", markersize=3, linewidth=1.5, label="BGE-M3 Abstract")
    bge_abs_best = df_bge_abstract.iloc[0]
    ax.plot(bge_abs_best["weight_dense"], bge_abs_best["Overall_Hit@10"],
            "*", color="#2171b5", markersize=12, zorder=5)

ps_best_a = df_ps_abstract.iloc[0]
ax.plot(ps_best_a["weight_dense"], ps_best_a["Overall_Hit@10"],
        "*", color="#cb181d", markersize=12, zorder=5)

ax.set_xlabel("Dense Weight (w_dense)")
ax.set_title("Abstract Field")
ax.legend(fontsize=7, loc="lower center")
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_xlim(-0.02, 1.02)

fig.suptitle("2-Way Hybrid Weight Curves: BGE-M3 vs PatentSBERTa", fontsize=10, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "issue3_weight_curves_comparison.png"))
fig.savefig(os.path.join(FIGURES_DIR, "issue3_weight_curves_comparison.pdf"))
plt.close(fig)
print("  Saved: issue3_weight_curves_comparison.png/pdf")

# Table 3: Per-model optimal weight comparison
tbl3_lines = []
tbl3_lines.append(f"{'Model × Field':<25s} | {'Opt w_d':>7s} | {'Opt w_b':>7s} | "
                  f"{'Opt Hit@10':>10s} | {'EqW Hit@10':>10s} | {'Δ(Opt-EqW)':>10s}")
tbl3_lines.append("-" * 85)
for _, row in df_summary_optimal.iterrows():
    tbl3_lines.append(f"{row['model_field']:<25s} | {row['optimal_w_dense']:7.2f} | "
                      f"{row['optimal_w_bm25']:7.2f} | {row['optimal_Hit@10']:10.4f} | "
                      f"{row['equal_weight_Hit@10']:10.4f} | {row['delta_optimal_vs_equal']:+10.4f}")
tbl3_text = "\n".join(tbl3_lines)
with open(os.path.join(TABLES_DIR, "issue3_optimal_weights_table.txt"), "w") as f:
    f.write(tbl3_text)
print("  Saved: issue3_optimal_weights_table.txt")
print(tbl3_text)


# ─────────────────────────────────────────────────────────────────────────────
# 이슈 4  Figure: WIPO sector별 성능 + hybrid improvement
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Generating Issue 4 figures & tables ---")

if "wipo_sector" in df_all.columns and df_all["wipo_sector"].notna().any():
    sectors = sorted(df_all["wipo_sector"].dropna().unique())
    sector_short = {
        "Electrical_engineering": "Electrical\nEng.",
        "Instruments":           "Instru-\nments",
        "Chemistry":             "Chemistry",
        "Mechanical_engineering": "Mechanical\nEng.",
        "Other_fields":          "Other\nFields",
    }
    
    # Figure 4-A: 주요 모델별 Hit@10 across sectors (grouped bar)
    fig, ax = plt.subplots(figsize=(7.0, 3.8))
    
    key_models = [
        ("BGE_M3_512_CLAIMS_HYBRID",   "BGE-M3 Hybrid",      "#2171b5"),
        ("PATENTSBERTA_CLAIMS_HYBRID", "PatentSBERTa Hybrid", "#cb181d"),
        ("BGE_M3_512_CLAIMS_DENSE",    "BGE-M3 Dense",        "#6baed6"),
        ("PATENTSBERTA_CLAIMS_DENSE",  "PatentSBERTa Dense",  "#fb6a4a"),
        ("BM25_CLAIMS",                "BM25 Claims",         "#74c476"),
    ]
    
    x = np.arange(len(sectors))
    n_models = len(key_models)
    bar_w = 0.15
    
    for m_idx, (mid, label, color) in enumerate(key_models):
        col = f"{mid}_hit10"
        if col not in df_all.columns:
            continue
        vals = []
        for s in sectors:
            sub = df_all[df_all["wipo_sector"] == s]
            vals.append(sub[col].mean())
        offset = (m_idx - n_models / 2 + 0.5) * bar_w
        ax.bar(x + offset, vals, bar_w, label=label, color=color, alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels([sector_short.get(s, s) for s in sectors], fontsize=7.5)
    ax.set_ylabel("Hit@10")
    ax.set_ylim(0.6, 1.0)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.legend(fontsize=7, ncol=3, loc="lower right", framealpha=0.9)
    ax.set_title("Hit@10 by WIPO Technology Sector (Claims Field)")
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "issue4_sector_performance.png"))
    fig.savefig(os.path.join(FIGURES_DIR, "issue4_sector_performance.pdf"))
    plt.close(fig)
    print("  Saved: issue4_sector_performance.png/pdf")
    
    # Figure 4-B: Hybrid improvement (Δ) by sector
    fig, ax = plt.subplots(figsize=(6.0, 3.2))
    
    improvement_pairs = [
        ("BGE_M3_512_CLAIMS_DENSE",    "BGE_M3_512_CLAIMS_HYBRID",    "BGE-M3 Claims",      "#2171b5"),
        ("PATENTSBERTA_CLAIMS_DENSE",  "PATENTSBERTA_CLAIMS_HYBRID",  "PatentSBERTa Claims", "#cb181d"),
    ]
    
    x = np.arange(len(sectors))
    bar_w = 0.3
    
    for p_idx, (dense_id, hybrid_id, label, color) in enumerate(improvement_pairs):
        d_col = f"{dense_id}_hit10"
        h_col = f"{hybrid_id}_hit10"
        if d_col not in df_all.columns or h_col not in df_all.columns:
            continue
        deltas = []
        for s in sectors:
            sub = df_all[df_all["wipo_sector"] == s]
            d_mean = sub[d_col].mean()
            h_mean = sub[h_col].mean()
            deltas.append(h_mean - d_mean)
        
        offset = (p_idx - 0.5) * bar_w
        bars = ax.bar(x + offset, [d * 100 for d in deltas], bar_w - 0.02,
                       label=label, color=color, alpha=0.8)
        # 값 표시
        for b_idx, bar in enumerate(bars):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                    f"+{deltas[b_idx]:.1%}", ha="center", va="bottom",
                    fontsize=6, color=color)
    
    ax.set_xticks(x)
    ax.set_xticklabels([sector_short.get(s, s) for s in sectors], fontsize=7.5)
    ax.set_ylabel("Δ Hit@10 (Hybrid − Dense, %p)")
    ax.set_title("Hybrid Improvement over Dense-Only by WIPO Sector")
    ax.legend(fontsize=7.5, loc="upper right")
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.axhline(0, color="gray", linewidth=0.5)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "issue4_sector_hybrid_improvement.png"))
    fig.savefig(os.path.join(FIGURES_DIR, "issue4_sector_hybrid_improvement.pdf"))
    plt.close(fig)
    print("  Saved: issue4_sector_hybrid_improvement.png/pdf")
    
    # Table 4: WIPO sector × key models Hit@10
    tbl4_lines = []
    header = f"{'Sector':<20s}"
    for _, label, _ in key_models:
        header += f" | {label:>16s}"
    header += f" | {'n_queries':>9s}"
    tbl4_lines.append(header)
    tbl4_lines.append("-" * len(header))
    
    for s in sectors:
        sub = df_all[df_all["wipo_sector"] == s]
        line = f"{s:<20s}"
        for mid, _, _ in key_models:
            col = f"{mid}_hit10"
            val = sub[col].mean() if col in sub.columns else float("nan")
            line += f" | {val:16.4f}"
        line += f" | {len(sub):9d}"
        tbl4_lines.append(line)
    
    tbl4_text = "\n".join(tbl4_lines)
    with open(os.path.join(TABLES_DIR, "issue4_sector_performance_table.txt"), "w") as f:
        f.write(tbl4_text)
    print("  Saved: issue4_sector_performance_table.txt")
    print(tbl4_text)

else:
    print("  [SKIP] WIPO sector figures skipped (no sector data)")


# =============================================================================
# 최종 요약
# =============================================================================
print("\n" + "=" * 70)
print("재실험코드 6 (NOTEBOOK 06) COMPLETE — Output Summary")
print("=" * 70)
print(f"\nAll outputs saved to: {OUT_DIR}")
print("\nGenerated files:")
for root, dirs, files in os.walk(OUT_DIR):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        rel = os.path.relpath(fpath, OUT_DIR)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {rel:60s} ({size_kb:.1f} KB)")

print("\n이슈별 핵심 출력 파일:")
print("  이슈 1 (Overlap & Lexical Shortcut):")
print("    CSV:    issue1_overlap_by_query_type.csv, issue1_lexical_shortcut_pivot.csv")
print("    Figure: figures/issue1_hybrid_improvement_by_overlap.png")
print("    Figure: figures/issue1_overlap_distribution_by_type.png")
print("    Table:  tables/issue1_overlap_statistics.txt")
print("    Table:  tables/issue1_lexical_shortcut_table.txt")
print("  이슈 2 (Statistical Tests):")
print("    CSV:    issue2_paired_bootstrap_results.csv")
print("    Figure: figures/issue2_forest_plot_bootstrap.png")
print("    Table:  tables/issue2_bootstrap_table.txt")
print("  이슈 3 (Weight Bias):")
print("    CSV:    issue3_gridsearch_patentsberta_claims.csv, issue3_per_model_optimal_weights.csv")
print("    Figure: figures/issue3_weight_curves_comparison.png")
print("    Table:  tables/issue3_optimal_weights_table.txt")
print("  이슈 4 (WIPO Sector):")
print("    CSV:    issue4_wipo_sector_pivot.csv, issue4_wipo_sector_hybrid_improvement.csv")
print("    Figure: figures/issue4_sector_performance.png")
print("    Figure: figures/issue4_sector_hybrid_improvement.png")
print("    Table:  tables/issue4_sector_performance_table.txt")
print("\n공통: query_level_hits_all.csv (전체 query-level 원시 데이터)")

재실험코드 6 (NOTEBOOK 06) — Supplementary Analyses

Phase 0: Building query-level hit matrix from rank caches

--- Year 2005 ---
  Queries loaded: 3150
  Corpus loaded: 141,170 docs (after dedup)
  Valid queries: 3150 / 3150
  BM25_Claims                         Hit@10=0.8867
  BM25_Abstract                       Hit@10=0.8733
  PatentSBERTa_Claims_Dense           Hit@10=0.8073
  PatentSBERTa_Abstract_Dense         Hit@10=0.8714
  PatentSBERTa_Claims_Hybrid          Hit@10=0.9140
  PatentSBERTa_Abstract_Hybrid        Hit@10=0.9232
  BGE-M3_512_Claims_Dense             Hit@10=0.8870
  BGE-M3_512_Abstract_Dense           Hit@10=0.9114
  BGE-M3_512_Claims_Hybrid            Hit@10=0.9371
  BGE-M3_512_Abstract_Hybrid          Hit@10=0.9311

--- Year 2015 ---
  Queries loaded: 3150
  Corpus loaded: 292,573 docs (after dedup)
  Valid queries: 3150 / 3150
  BM25_Claims                         Hit@10=0.8810
  BM25_Abstract                       Hit@10=0.8086
  PatentSBERTa_Claims_Dense           Hi

C:\Users\rokk99\AppData\Local\Temp\ipykernel_10856\1785028385.py:1128: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(qt_data, labels=["Summary", "Keyword", "Function"],


  Saved: issue2_forest_plot_bootstrap.png/pdf

--- Generating Issue 3 figures & tables ---
  [INFO] Running BGE-M3 Abstract 2-way grid search...
  Saved: issue3_weight_curves_comparison.png/pdf
  Saved: issue3_optimal_weights_table.txt
Model × Field             | Opt w_d | Opt w_b | Opt Hit@10 | EqW Hit@10 | Δ(Opt-EqW)
-------------------------------------------------------------------------------------
PatentSBERTa_Claims       |    0.35 |    0.65 |     0.9050 |     0.9004 |    +0.0046
PatentSBERTa_Abstract     |    0.50 |    0.50 |     0.8842 |     0.8842 |    +0.0000
BGE-M3_Claims             |    0.50 |    0.50 |     0.9216 |     0.9216 |    +0.0000

--- Generating Issue 4 figures & tables ---
  Saved: issue4_sector_performance.png/pdf
  Saved: issue4_sector_hybrid_improvement.png/pdf
  Saved: issue4_sector_performance_table.txt
Sector               |    BGE-M3 Hybrid | PatentSBERTa Hybrid |     BGE-M3 Dense | PatentSBERTa Dense |      BM25 Claims | n_queries
----------------------